# RAG Pipeline Exploration

End-to-end test of the Multi-Doc RAG Q&A system on real PDFs.

**Pipeline:** PDF → Pages → Chunks → Embeddings → Vector DB → Retrieval → LLM → Answer + Citations

## Setup
1. Drop 2-5 PDFs into `data/sample_pdfs/`
2. Run the cells in order
3. Ask questions in the final cell

In [1]:
import sys
sys.path.append('..')

from src.pdf_loader import load_pdfs
from src.chunker import chunk_pages, DEFAULT_CHUNK_SIZE, DEFAULT_CHUNK_OVERLAP
from src.vectorstore import ingest_chunks, clear_vectorstore, get_vectorstore, DEFAULT_DB_PATH
from src.chain import ask

print('✓ All modules imported')

✓ All modules imported


## Step 1: Load PDFs

Drop your PDFs into `data/sample_pdfs/` first. Then run the cell below.

In [2]:
from pathlib import Path

pdf_dir = Path('../data/sample_pdfs')
pdf_files = sorted(pdf_dir.glob('*.pdf'))

if not pdf_files:
    print('⚠ No PDFs found in data/sample_pdfs/')
    print('  Drop some PDFs there, then re-run this cell.')
else:
    print(f'Found {len(pdf_files)} PDFs:')
    for p in pdf_files:
        print(f'  - {p.name}')

Found 1 PDFs:
  - 20220420_Air60 Quick Guide.pdf


In [3]:
pages = load_pdfs([str(p) for p in pdf_files])
print(f'\nTotal pages loaded: {len(pages)}')

    → Page 1: no text found, trying OCR...
    ✓ OCR extracted 4321 chars from page 1
  ✓ Loaded 1 pages from 20220420_Air60 Quick Guide.pdf (0 text, 1 OCR)

Total pages loaded: 1


## Step 2: Chunk the pages

In [4]:
chunks = chunk_pages(pages, chunk_size=DEFAULT_CHUNK_SIZE, chunk_overlap=DEFAULT_CHUNK_OVERLAP)
print(f'\nTotal chunks: {len(chunks)}')

  ✓ 20220420_Air60 Quick Guide.pdf p.1: 6 chunks

Total chunks: 6


## Step 3: Ingest into vector DB

First run takes ~1 sec/chunk. Subsequent runs skip (persistent DB).

In [5]:
count = ingest_chunks(chunks, db_path=DEFAULT_DB_PATH)
print(f'Ingested {count} chunks')

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


  ✓ Embedding model loaded (384-dim vectors, CPU)
  ✓ Ingested 6 chunks (1 files) → ./chroma_db
Ingested 6 chunks


## Step 4: Ask questions

Replace the question below with your own and re-run.

In [6]:
result = ask(
    question='What is this document about?',
    db_path=DEFAULT_DB_PATH,
    top_k=4,
)

print('Question:', result.question)
print('\nAnswer:', result.answer)
print('\nCitations:')
for i, c in enumerate(result.citations, 1):
    print(f'  [{i}] {c.citation} (score: {c.score:.4f})')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



  [1/3] Retrieving top-4 chunks for: 'What is this document about?...'
       Found 4 chunks
  [2/3] Building prompt with context...
  [3/3] Generating answer with LLM...
Initializing LLM: openai/gpt-oss-120b (temp=0.1)...
  ✓ LLM ready (openai/gpt-oss-120b)
       ✓ Answer generated (259 chars)
Question: What is this document about?

Answer: The document is a quick‑reference guide for the Air60 keyboard, detailing key‑combination shortcuts for functions such as factory reset, battery‑level indicators, sleep mode, sidelight effects, and wireless (Bluetooth/2.4 GHz) device connections. [1][2][3][4]

Citations:
  [1] 20220420_Air60 Quick Guide.pdf, p.1 (score: 0.3690)
  [2] 20220420_Air60 Quick Guide.pdf, p.1 (score: 0.3543)
  [3] 20220420_Air60 Quick Guide.pdf, p.1 (score: 0.3533)
  [4] 20220420_Air60 Quick Guide.pdf, p.1 (score: 0.3507)


## Step 5: Try more questions

Add your own questions below.

In [7]:
my_questions = [
    'What skills are mentioned?',
    'What is the main topic?',
]

for q in my_questions:
    result = ask(question=q, db_path=DEFAULT_DB_PATH, top_k=4)
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    print(f'A: {result.answer}')
    print(f'Citations: {[c.citation for c in result.citations]}')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



  [1/3] Retrieving top-4 chunks for: 'What skills are mentioned?...'
       Found 4 chunks
  [2/3] Building prompt with context...
  [3/3] Generating answer with LLM...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


       ✓ Answer generated (43 chars)

Q: What skills are mentioned?
A: I don't know based on the provided context.
Citations: ['20220420_Air60 Quick Guide.pdf, p.1', '20220420_Air60 Quick Guide.pdf, p.1', '20220420_Air60 Quick Guide.pdf, p.1', '20220420_Air60 Quick Guide.pdf, p.1']

  [1/3] Retrieving top-4 chunks for: 'What is the main topic?...'
       Found 4 chunks
  [2/3] Building prompt with context...
  [3/3] Generating answer with LLM...
       ✓ Answer generated (177 chars)

Q: What is the main topic?
A: The provided text is a quick‑guide/manual for the Air60 keyboard, describing its functions (factory reset, key mappings, connection modes, LED and battery settings, etc.)【1】【2】.
Citations: ['20220420_Air60 Quick Guide.pdf, p.1', '20220420_Air60 Quick Guide.pdf, p.1', '20220420_Air60 Quick Guide.pdf, p.1', '20220420_Air60 Quick Guide.pdf, p.1']


## Cleanup (optional)

Run this to wipe the vector DB (e.g., before re-ingesting new PDFs).

In [ ]:
# clear_vectorstore(db_path=DEFAULT_DB_PATH)
# print('Vector DB cleared. Re-run Step 3 to re-ingest.')